In [1]:
import itertools
import numpy as np
import gurobipy as gp
from gurobipy import GRB



In [41]:
def balanced_fair_cut(hyperedges,weights,n):
    assert len(weights)==len(edges)
    model = gp.Model()
    model.Params.OutputFlag=0

    target_ones = (n+1)//2
    partitions=[]

    for ones in itertools.combinations(range(n),target_ones):
        x = np.zeros(n,dtype=int)
        for one in ones: 
            x[one]=1
        partitions.append(x)

    p = model.addVars(len(partitions),lb=0,ub=1,name="p")
    t = model.addVar(lb=0,name='t')

    model.addConstr(
        gp.quicksum(p[j] for j in range(len(partitions))) == 1,
        name="prob_normalization"
    )#sum probabilities=1
    
    # Weighted fair cut constraints
    for e_idx, edge in enumerate(edges):
        expected_cut = 0
        for j, partition in enumerate(partitions):
            vertex_assignments = [partition[v] for v in edge]#list of the vertices in each hyperedge
            is_cut = int(len(set(vertex_assignments)) > 1)
            expected_cut += p[j] * is_cut 
        # weighted probability of cutting this hyperedge >= t
        model.addConstr(weights[e_idx] * expected_cut >= t,name=f"coverage_{e_idx}")
    model.setObjective(t,GRB.MAXIMIZE)
    model.optimize()
    assert model.Status == GRB.OPTIMAL, (f"Gurobi did not prove optimality. Status = {model.Status}")
    probs =np.array([p[j].X for j in range(len(partitions))]) #get solution value from gurobi 
    cut_probabilities = []

    # Calculate the probability of cutting each hyperedge
    for edge in edges:
        cut_probability = 0
        for j, partition in enumerate(partitions):
            vertex_assignments = [partition[v] for v in edge]
            is_cut = int(len(set(vertex_assignments)) > 1)
            cut_probability += probs[j] * is_cut
    
        cut_probabilities.append(cut_probability)

    return (
        np.array(partitions),
        probs,
        cut_probabilities,
        model.ObjVal
    )
    
    

In [42]:
n = 6

edges = [
    (0, 1, 2),
    (3, 4, 5),
    (0, 3)
]
weights = [1,1,1]

In [43]:
balanced_fair_cut(edges,weights,n)

(array([[1, 1, 1, 0, 0, 0],
        [1, 1, 0, 1, 0, 0],
        [1, 1, 0, 0, 1, 0],
        [1, 1, 0, 0, 0, 1],
        [1, 0, 1, 1, 0, 0],
        [1, 0, 1, 0, 1, 0],
        [1, 0, 1, 0, 0, 1],
        [1, 0, 0, 1, 1, 0],
        [1, 0, 0, 1, 0, 1],
        [1, 0, 0, 0, 1, 1],
        [0, 1, 1, 1, 0, 0],
        [0, 1, 1, 0, 1, 0],
        [0, 1, 1, 0, 0, 1],
        [0, 1, 0, 1, 1, 0],
        [0, 1, 0, 1, 0, 1],
        [0, 1, 0, 0, 1, 1],
        [0, 0, 1, 1, 1, 0],
        [0, 0, 1, 1, 0, 1],
        [0, 0, 1, 0, 1, 1],
        [0, 0, 0, 1, 1, 1]]),
 array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.]),
 [np.float64(1.0), np.float64(1.0), np.float64(1.0)],
 1.0)

In [44]:
n = 4

edges = [
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 2),
    (1, 3),
    (2, 3)
]

weights = [1] * len(edges)

balanced_fair_cut(
    edges, weights, n
)

(array([[1, 1, 0, 0],
        [1, 0, 1, 0],
        [1, 0, 0, 1],
        [0, 1, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 1]]),
 array([0.        , 0.33333333, 0.33333333, 0.        , 0.        ,
        0.33333333]),
 [np.float64(0.6666666666666667),
  np.float64(0.6666666666666667),
  np.float64(0.6666666666666667),
  np.float64(0.6666666666666667),
  np.float64(0.6666666666666667),
  np.float64(0.6666666666666667)],
 0.6666666666666667)

In [48]:
def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges

In [50]:

import os
import glob
import numpy as np
import pandas as pd

poisson_types = [
    "poisson_16_12_3",
    "poisson_16_12_4",
    "poisson_16_12_5"
]

poisson_results = []

for graph_type in poisson_types:

    # Data folders are one level above extension/
    p_folder = f"../data/{graph_type}/P"
    w_folder = f"../data/{graph_type}/w"

    p_files = sorted(glob.glob(f"{p_folder}/*.npy"))

    for p_file in p_files:
        instance = os.path.splitext(os.path.basename(p_file))[0]

        # Load stochastic incidence matrix
        P = np.load(p_file)
        n = P.shape[0]
        edges = get_edge_list(P)

        # Load hyperedge weights
        w_file = os.path.join(w_folder, f"{instance}.npy")
        weights = np.load(w_file)
        weights = np.asarray(weights).reshape(-1)

        assert len(weights) == len(edges), (
            f"{graph_type}/{instance}: "
            f"{len(weights)} weights different than "
            f"{len(edges)} edges"
        )

        # Solve weighted balanced fair cut cover LP
        partitions, probs, cut_probs, optimal_t = (
            balanced_fair_cut(edges, weights, n)
        )

        # Weighted coverage of each hyperedge
        weighted_coverage = weights * cut_probs

        # Number of partitions assigned positive probability
        num_support = int(np.sum(probs > 1e-8))

        # Minimum unweighted cut probability
        min_cut_probability = float(np.min(cut_probs))

        poisson_results.append({
            "dataset": graph_type,
            "instance": instance,
            "n_vertices": n,
            "n_edges": len(edges),
            "optimal_t": optimal_t,
            "min_cut_probability": min_cut_probability,
            "num_support_partitions": num_support,
            "cut_probabilities": cut_probs,
            "weighted_coverages": weighted_coverage.tolist()
        })

        print(
            graph_type,
            instance,
            "| optimal t =", round(optimal_t, 6),
            "| min cut probability =", round(min_cut_probability, 6),
            "| support =", num_support
        )

# Convert results to DataFrame
poisson_df = pd.DataFrame(poisson_results)

display(poisson_df)


poisson_16_12_3 01 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 02 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 03 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 04 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 05 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 06 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 07 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 08 | optimal t = 1.0 | min cut probability = 0.5 | support = 2
poisson_16_12_3 09 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 10 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 11 | optimal t = 0.666667 | min cut probability = 0.666667 | support = 5
poisson_16_12_3 12 | optimal t = 1.0 | min cut probability = 1.0 | support = 1
poisson_16_12_3 13 | optimal t = 1.0 | min

,dataset,instance,n_vertices,n_edges,optimal_t,min_cut_probability,num_support_partitions,cut_probabilities,weighted_coverages
0,poisson_16_12_3,01,12,15,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,poisson_16_12_3,02,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,poisson_16_12_3,03,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
3,poisson_16_12_3,04,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
4,poisson_16_12_3,05,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
5,poisson_16_12_3,06,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
6,poisson_16_12_3,07,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
7,poisson_16_12_3,08,12,15,1.000000,0.500000,2,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
8,poisson_16_12_3,09,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
9,poisson_16_12_3,10,12,16,1.000000,1.000000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."


In [51]:

from collections import Counter

for graph_type in poisson_types:
    p_files = sorted(glob.glob(f"../data/{graph_type}/P/*.npy"))

    sizes = []

    for p_file in p_files:
        P = np.load(p_file)
        edges = get_edge_list(P)

        sizes.extend(len(edge) for edge in edges)

    print(graph_type)
    print("Hyperedge sizes:", dict(sorted(Counter(sizes).items())))
    print("Average size:", np.mean(sizes))
    print()


poisson_16_12_3
Hyperedge sizes: {2: 99, 3: 128, 4: 59, 5: 26, 6: 1, 7: 2}
Average size: 3.073015873015873

poisson_16_12_4
Hyperedge sizes: {2: 20, 3: 41, 4: 49, 5: 27, 6: 13, 7: 8, 8: 1}
Average size: 4.0

poisson_16_12_5
Hyperedge sizes: {2: 9, 3: 19, 4: 31, 5: 43, 6: 27, 7: 13, 8: 9, 9: 3, 10: 4, 11: 1}
Average size: 5.138364779874214

